# Chapter 10: Advanced Architectures & Efficiency

Modern LLMs are not just bigger transformers — they are engineered systems combining architectural innovations and systems-level optimizations. This chapter derives the mathematics behind each efficiency technique and implements it from scratch.

**Topics covered:**
1. Mixture of Experts (MoE)
2. Grouped Query Attention (GQA)
3. INT8 Quantization
4. Speculative Decoding
5. KV Cache
6. Knowledge Distillation
7. Magnitude Pruning
8. Efficient Inference Stack
9. Summary: Math → Engineering

**Install:** `!pip install torch` if needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Mixture of Experts (MoE)

MoE replaces the dense feed-forward network (FFN) in a transformer with a **mixture of $N$ expert FFNs**, activating only the top-$k$ for each token.

### Router (Gating Mechanism)

$$\mathbf{g}(\mathbf{x}) = \text{TopK}\!\left(\text{softmax}\!\left(\mathbf{x}\mathbf{W}_g\right), k\right)$$

where $\mathbf{W}_g \in \mathbb{R}^{d \times N}$ produces routing logits. Only the top-$k$ logits (and their softmax weights) are kept; all others are zeroed out.

### Output Computation

$$\text{MoE}(\mathbf{x}) = \sum_{i \in \text{TopK}} g_i(\mathbf{x}) \cdot E_i(\mathbf{x})$$

Each expert $E_i$ is a standard FFN: $E_i(\mathbf{x}) = \text{GELU}(\mathbf{x}\mathbf{W}_1^{(i)})\mathbf{W}_2^{(i)}$.

### Load Balancing Loss

Without regularization, the router collapses (all tokens sent to 1-2 experts). The auxiliary loss:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N \sum_{i=1}^{N} f_i \cdot P_i$$

where $f_i = \frac{1}{T}\sum_t \mathbf{1}[i \in \text{TopK}(t)]$ (fraction of tokens routed to expert $i$) and $P_i = \frac{1}{T}\sum_t g_i(\mathbf{x}_t)$ (mean routing probability for expert $i$).

**Efficiency gain:** $N = 8$ experts with $k = 2$ active means only $2/8 = 25\%$ of FFN parameters are computed per token, but the model has $8 \times$ the FFN capacity.

In [ ]:
class MoELayer(nn.Module):
    """Mixture of Experts FFN layer with top-k routing and load balance loss."""

    def __init__(self, d_model=64, d_ff=128, num_experts=8, top_k=2, load_balance_alpha=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.load_balance_alpha = load_balance_alpha

        # Router: maps d_model -> num_experts logits
        self.router = nn.Linear(d_model, num_experts, bias=False)

        # Expert FFNs: each is a 2-layer MLP
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff, bias=False),
                nn.GELU(),
                nn.Linear(d_ff, d_model, bias=False)
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        """
        Args:
            x: (B, T, d_model)
        Returns:
            output: (B, T, d_model)
            aux_loss: load balancing scalar loss
            routing_info: dict with expert load statistics
        """
        B, T, d = x.shape
        x_flat = x.view(B * T, d)   # (B*T, d)

        # Router logits and softmax
        router_logits = self.router(x_flat)              # (B*T, N)
        router_probs  = F.softmax(router_logits, dim=-1) # (B*T, N)

        # Top-k selection
        topk_probs, topk_indices = router_probs.topk(self.top_k, dim=-1)  # (B*T, k)
        # Renormalize top-k weights
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)    # (B*T, k)

        # Compute expert outputs and combine
        output = torch.zeros(B * T, d, device=x.device, dtype=x.dtype)
        expert_counts = torch.zeros(self.num_experts, device=x.device)

        for k_idx in range(self.top_k):
            expert_idx = topk_indices[:, k_idx]   # (B*T,) which expert
            weights    = topk_probs[:, k_idx]      # (B*T,) routing weight

            for e in range(self.num_experts):
                token_mask = (expert_idx == e)     # which tokens go to expert e
                if token_mask.any():
                    tokens_for_expert = x_flat[token_mask]       # (n_e, d)
                    expert_out = self.experts[e](tokens_for_expert)  # (n_e, d)
                    output[token_mask] += weights[token_mask].unsqueeze(-1) * expert_out
                    expert_counts[e] += token_mask.sum()

        # Load balancing loss
        # f_i: fraction of tokens dispatched to expert i
        f = expert_counts / (B * T * self.top_k)
        # P_i: mean routing probability for expert i
        P = router_probs.mean(dim=0)
        aux_loss = self.load_balance_alpha * self.num_experts * (f * P).sum()

        return output.view(B, T, d), aux_loss, {"expert_counts": expert_counts, "load_fracs": f}


torch.manual_seed(42)
B, T, d_model = 2, 6, 64
moe = MoELayer(d_model=d_model, d_ff=128, num_experts=8, top_k=2)

x = torch.randn(B, T, d_model)
out, aux_loss, routing_info = moe(x)

print(f"MoE Layer: 8 experts, top-2 routing")
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Aux loss (load balance): {aux_loss.item():.6f}")
print(f"Expert token counts: {routing_info['expert_counts'].tolist()}")
print(f"Load fractions f_i:  {routing_info['load_fracs'].round(decimals=3).tolist()}")

# Verify only 2 experts active per token (by examining routing)
router_logits = moe.router(x.view(-1, d_model))
_, topk_idx = router_logits.topk(2, dim=-1)
total_tokens = B * T
print(f"\nVerification: Each of {total_tokens} tokens uses exactly {moe.top_k} experts")
print(f"  Unique active experts per token: {topk_idx.shape[1]} (always == top_k={moe.top_k})")

# Parameter count comparison: dense FFN vs MoE FFN
d_ff = 128
dense_ffn_params = 2 * d_model * d_ff
moe_total_params = 8 * 2 * d_model * d_ff + d_model * 8  # 8 experts + router
moe_active_params = 2 * 2 * d_model * d_ff               # top-2 experts active
print(f"\nParameter comparison (d_model={d_model}, d_ff={d_ff}):")
print(f"  Dense FFN total/active:  {dense_ffn_params:,} / {dense_ffn_params:,}")
print(f"  MoE total / active:      {moe_total_params:,} / {moe_active_params:,}")
print(f"  MoE capacity multiplier: {moe_total_params/dense_ffn_params:.1f}x with {moe_active_params/dense_ffn_params:.2f}x compute")

## 2. Grouped Query Attention (GQA)

KV cache at inference scales as $O(L \cdot T \cdot h \cdot d_k)$ — with 70B parameter models and long contexts, this dominates memory.

### Attention Variants

| Variant | Q heads | K/V heads | KV cache size |
|---|---|---|---|
| **MHA** (standard) | $H$ | $H$ | $2 \cdot L \cdot T \cdot H \cdot d_k$ |
| **MQA** | $H$ | 1 | $2 \cdot L \cdot T \cdot d_k$ |
| **GQA** | $H$ | $G$ | $2 \cdot L \cdot T \cdot G \cdot d_k$ |

GQA groups query heads: $G$ K/V heads are each shared by $H/G$ query heads. LLaMA-3 uses $H=32$ query heads with $G=8$ KV heads — a $4\times$ KV cache reduction.

### KV Cache Memory Formula

$$\text{KV bytes} = 2 \times L \times T \times G \times d_k \times \text{bytes\_per\_element}$$

For LLaMA-3 70B (BF16): $L=80$, $T=4096$, $G=8$, $d_k=128$ → $2 \times 80 \times 4096 \times 8 \times 128 \times 2 \approx 16.8$ GB

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Multi-head attention with grouped key/value heads (GQA)."""

    def __init__(self, d_model=64, num_heads=8, num_kv_heads=2):
        super().__init__()
        assert num_heads % num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"
        self.num_heads    = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups   = num_heads // num_kv_heads   # heads per KV group
        self.d_head       = d_model // num_heads
        self.d_kv_head    = d_model // num_heads        # KV head dim same as Q head dim

        # Q projection: num_heads full heads
        self.q_proj = nn.Linear(d_model, num_heads * self.d_head, bias=False)
        # K, V projections: only num_kv_heads
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.d_kv_head, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.d_kv_head, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, _ = x.shape

        # Project and reshape
        Q = self.q_proj(x).view(B, T, self.num_heads,    self.d_head).transpose(1, 2)    # (B, H, T, d_head)
        K = self.k_proj(x).view(B, T, self.num_kv_heads, self.d_kv_head).transpose(1, 2) # (B, G, T, d_head)
        V = self.v_proj(x).view(B, T, self.num_kv_heads, self.d_kv_head).transpose(1, 2) # (B, G, T, d_head)

        # Expand K, V to match num_heads via repeat_interleave
        # Each KV head is shared by (num_heads / num_kv_heads) query heads
        K = torch.repeat_interleave(K, self.num_groups, dim=1)   # (B, H, T, d_head)
        V = torch.repeat_interleave(V, self.num_groups, dim=1)   # (B, H, T, d_head)

        # Scaled dot-product attention
        scale = math.sqrt(self.d_head)
        scores = (Q @ K.transpose(-2, -1)) / scale                 # (B, H, T, T)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = attn @ V                                              # (B, H, T, d_head)

        # Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, -1)      # (B, T, d_model)
        return self.o_proj(out)


torch.manual_seed(42)
B, T, d_model = 2, 8, 64
gqa = GroupedQueryAttention(d_model=d_model, num_heads=8, num_kv_heads=2)

x = torch.randn(B, T, d_model)
out = gqa(x)
print(f"GQA (num_heads=8, num_kv_heads=2, groups=4)")
print(f"Input:  {x.shape}  ->  Output: {out.shape}")

# Parameter counts
def count_qkv_params(d_model, num_heads, num_kv_heads):
    d_head = d_model // num_heads
    q_params = d_model * num_heads * d_head
    kv_params = 2 * d_model * num_kv_heads * d_head
    o_params = d_model * d_model
    return q_params, kv_params, o_params

print("\nQKV parameter comparison (d_model=512, d_head=64):")
d_model_cmp = 512
for name, nh, nkv in [("MHA", 8, 8), ("GQA (G=4)", 8, 4), ("GQA (G=2)", 8, 2), ("MQA", 8, 1)]:
    q_p, kv_p, o_p = count_qkv_params(d_model_cmp, nh, nkv)
    total = q_p + kv_p + o_p
    print(f"  {name:12s}: Q={q_p:>8,}  KV={kv_p:>8,}  O={o_p:>8,}  total={total:>10,}")

# KV cache memory for various configs
print("\nKV cache memory (bytes, BF16=2B per element):")
print(f"{'Config':>20} | {'L':>4} | {'T':>6} | {'G':>4} | {'d_k':>5} | {'KV cache GB':>12}")
print("-" * 65)
for name, L, T_seq, G, d_k in [
    ("LLaMA-3 8B MHA",  32,  4096, 32, 128),
    ("LLaMA-3 8B GQA",  32,  4096,  8, 128),
    ("LLaMA-3 70B GQA", 80,  4096,  8, 128),
    ("LLaMA-3 70B GQA", 80, 32768,  8, 128),
]:
    kv_bytes = 2 * L * T_seq * G * d_k * 2   # 2 for K+V, 2 bytes for BF16
    kv_gb = kv_bytes / 1e9
    print(f"  {name:>20} | {L:>4} | {T_seq:>6} | {G:>4} | {d_k:>5} | {kv_gb:>12.2f} GB")

## 3. INT8 Quantization

Quantization reduces weight precision from FP32/BF16 to INT8 (or INT4), halving (or quartering) memory bandwidth.

### Per-Channel Quantization

For weight matrix $\mathbf{W} \in \mathbb{R}^{d_{out} \times d_{in}}$, compute one scale per output channel:

$$s_i = \frac{\max(|W_{i,:}|)}{127}, \quad W_q[i, j] = \text{round}\!\left(\frac{W[i, j]}{s_i}\right)$$

Dequantization at inference:

$$W_{\text{dequant}}[i, j] = s_i \cdot W_q[i, j]$$

### GPTQ (Post-Training Quantization)

GPTQ (Frantar et al., 2022) minimizes the second-order reconstruction error:

$$\min_{\hat{\mathbf{W}}} \|\mathbf{W}\mathbf{X} - \hat{\mathbf{W}}\mathbf{X}\|_F^2$$

by sequentially quantizing columns and compensating remaining columns using the inverse Hessian $\mathbf{H}^{-1} = (\mathbf{X}\mathbf{X}^T)^{-1}$.

### Memory Savings

| Precision | Bits | Bytes per param | 7B model |
|---|---|---|---|
| FP32 | 32 | 4 | 28 GB |
| BF16/FP16 | 16 | 2 | 14 GB |
| INT8 | 8 | 1 | 7 GB |
| INT4 | 4 | 0.5 | 3.5 GB |

In [ ]:
class QuantizedLinear(nn.Module):
    """Per-channel INT8 quantized linear layer."""

    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # FP32 reference weight (for initialization; normally loaded from checkpoint)
        fp32_weight = torch.randn(out_features, in_features) * 0.02

        # Per-channel scale: one per output channel
        scales = fp32_weight.abs().max(dim=1).values / 127.0  # (out_features,)
        scales = scales.clamp(min=1e-8)

        # Quantize: round to nearest INT8
        weight_q = (fp32_weight / scales.unsqueeze(1)).round().clamp(-127, 127).to(torch.int8)

        # Store as non-parameter buffers (INT8 for memory, scales FP32)
        self.register_buffer('weight_q', weight_q)
        self.register_buffer('scales', scales)

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.bias = None

        # Save FP32 reference for error comparison
        self.register_buffer('weight_fp32', fp32_weight)

    def dequantize(self):
        """Reconstruct approximate FP32 weight from INT8."""
        return self.weight_q.float() * self.scales.unsqueeze(1)

    def forward(self, x):
        # Dequantize at inference (in practice done in fused kernel)
        weight_deq = self.dequantize()
        out = F.linear(x, weight_deq, self.bias)
        return out

    def memory_bytes(self):
        int8_bytes = self.weight_q.numel() * 1         # 1 byte per INT8
        scale_bytes = self.scales.numel() * 4           # 4 bytes per FP32 scale
        fp32_bytes  = self.out_features * self.in_features * 4  # hypothetical FP32
        return {"int8": int8_bytes + scale_bytes, "fp32": fp32_bytes}


torch.manual_seed(42)
d_in, d_out = 512, 256
q_layer = QuantizedLinear(d_in, d_out)

# Compare output to FP32 reference
x_test = torch.randn(4, d_in)
out_int8 = q_layer(x_test)
out_fp32 = F.linear(x_test, q_layer.weight_fp32, q_layer.bias)

abs_err = (out_int8 - out_fp32).abs()
rel_err = abs_err / (out_fp32.abs() + 1e-8)

print(f"INT8 Quantized Linear ({d_in} -> {d_out}):")
print(f"  Max absolute error: {abs_err.max().item():.6f}")
print(f"  Mean absolute error: {abs_err.mean().item():.6f}")
print(f"  Mean relative error: {rel_err.mean().item():.4%}")

# Weight reconstruction error
w_deq = q_layer.dequantize()
w_err = (w_deq - q_layer.weight_fp32).abs()
print(f"  Weight reconstruction max error: {w_err.max().item():.6f}")
print(f"  Weight reconstruction mean error: {w_err.mean().item():.6f}")

# Memory comparison
mem = q_layer.memory_bytes()
print(f"\nMemory:")
print(f"  FP32:  {mem['fp32']:,} bytes  ({mem['fp32']/1024:.1f} KB)")
print(f"  INT8:  {mem['int8']:,} bytes  ({mem['int8']/1024:.1f} KB)")
print(f"  Ratio: {mem['fp32']/mem['int8']:.2f}x compression")

# Show quantization across model sizes
print("\nMemory for various LLM sizes:")
for name, n_params in [("7B", 7e9), ("13B", 13e9), ("70B", 70e9), ("405B", 405e9)]:
    fp32_gb  = n_params * 4 / 1e9
    bf16_gb  = n_params * 2 / 1e9
    int8_gb  = n_params * 1 / 1e9
    int4_gb  = n_params * 0.5 / 1e9
    print(f"  {name:>6}: FP32={fp32_gb:>7.1f} GB  BF16={bf16_gb:>7.1f} GB  INT8={int8_gb:>7.1f} GB  INT4={int4_gb:>7.1f} GB")

## 4. Speculative Decoding

Speculative decoding (Leviathan et al., 2023; Chen et al., 2023) dramatically speeds up inference by using a **small draft model** to propose multiple tokens, then verifying them in a single forward pass of the larger target model.

### Algorithm

1. **Draft phase:** Small model auto-regressively generates $\gamma$ candidate tokens $\tilde{x}_1, \ldots, \tilde{x}_\gamma$ with probabilities $q(\tilde{x}_t)$
2. **Verify phase:** Target model scores all $\gamma+1$ positions in **one forward pass**, producing $p(x \mid \text{context})$
3. **Acceptance sampling:** Accept token $\tilde{x}_t$ with probability:

$$\alpha_t = \min\!\left(1, \frac{p(\tilde{x}_t)}{q(\tilde{x}_t)}\right)$$

4. If rejected, sample from corrected distribution $p'(x) \propto \max(0, p(x) - q(x))$

### Expected Tokens per Step

$$\mathbb{E}[\text{accepted tokens}] = \sum_{t=1}^{\gamma} \prod_{s=1}^{t} \alpha_s \leq \gamma$$

When draft and target models are aligned ($p \approx q$), acceptance rate $\alpha \to 1$ and we get close to $\gamma$ tokens per target forward pass — a $\gamma \times$ speedup in memory-bandwidth-bound inference.

In [ ]:
def speculative_decode(draft_logits, target_logits, gamma=4, temperature=1.0):
    """Speculative decoding with rejection sampling.
    Args:
        draft_logits:  (gamma, V) logits from draft model for each of gamma positions
        target_logits: (gamma+1, V) logits from target model for gamma+1 positions
        gamma:         number of draft tokens
        temperature:   sampling temperature
    Returns:
        accepted_tokens: list of accepted token IDs
        acceptance_rate: fraction of draft tokens accepted
        final_token: the guaranteed new token (from target at rejection/end)
    """
    V = draft_logits.shape[-1]

    # Convert to probabilities
    q_probs = F.softmax(draft_logits / temperature, dim=-1)   # (gamma, V)
    p_probs = F.softmax(target_logits / temperature, dim=-1)  # (gamma+1, V)

    # Sample draft tokens
    draft_tokens = torch.multinomial(q_probs, num_samples=1).squeeze(-1)  # (gamma,)

    accepted_tokens = []
    final_accept_pos = 0

    for t in range(gamma):
        x_t = draft_tokens[t].item()

        # Acceptance probability: min(1, p(x_t) / q(x_t))
        p_xt = p_probs[t, x_t].item()
        q_xt = q_probs[t, x_t].item() + 1e-10  # numerical stability
        alpha = min(1.0, p_xt / q_xt)

        # Accept/reject
        u = torch.rand(1).item()
        if u <= alpha:
            accepted_tokens.append(x_t)
            final_accept_pos = t + 1
        else:
            # Sample from corrected distribution p'(x) ∝ max(0, p(x) - q(x))
            corrected = torch.clamp(p_probs[t] - q_probs[t], min=0.0)
            if corrected.sum() > 0:
                corrected = corrected / corrected.sum()
                final_token = torch.multinomial(corrected, 1).item()
            else:
                final_token = torch.multinomial(p_probs[t], 1).item()
            accepted_tokens.append(final_token)
            acceptance_rate = len([t for t in range(gamma) if t < final_accept_pos]) / gamma
            return accepted_tokens, acceptance_rate, final_token

    # All draft tokens accepted; sample one more from target at final position
    final_token = torch.multinomial(p_probs[gamma], 1).item()
    accepted_tokens.append(final_token)
    return accepted_tokens, 1.0, final_token


torch.manual_seed(42)
V = 100
gamma = 4

# Simulate well-aligned models (similar logits -> high acceptance)
base_logits = torch.randn(gamma + 1, V)

print("Speculative Decoding Acceptance Rate vs Model Alignment:")
print(f"{'Noise level':>12} | {'Acceptance rate':>16} | {'Tokens/step':>12} | {'Draft tokens'}")
print("-" * 65)

for noise in [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]:
    rates = []
    tokens_per_step = []
    for trial in range(200):
        torch.manual_seed(trial)
        draft_logits  = base_logits[:gamma] + torch.randn(gamma, V) * noise
        target_logits = base_logits
        toks, rate, _ = speculative_decode(draft_logits, target_logits, gamma=gamma)
        rates.append(rate)
        tokens_per_step.append(len(toks))

    mean_rate = sum(rates) / len(rates)
    mean_toks = sum(tokens_per_step) / len(tokens_per_step)
    print(f"{noise:>12.1f} | {mean_rate:>16.3f} | {mean_toks:>12.2f} | (gamma={gamma})")

print("\nAt noise=0.0, draft==target -> all accepted -> gamma+1 tokens per target pass")
print("At high noise, draft diverges from target -> low acceptance -> ~1 token per pass")

## 5. KV Cache

During autoregressive generation, re-computing K and V for all past tokens at every step is wasteful. The **KV cache** stores past keys and values, enabling $O(1)$ incremental attention.

### Memory Analysis

Per-token KV cache memory (across all layers):

$$\text{bytes per token} = 2 \times L \times G \times d_k \times \text{dtype\_bytes}$$

where the leading 2 accounts for K and V.

### For LLaMA-3 70B (BF16, GQA)

$L=80$ layers, $G=8$ KV heads, $d_k=128$, BF16 = 2 bytes:

$$\text{bytes/token} = 2 \times 80 \times 8 \times 128 \times 2 = 327{,}680 \text{ bytes} \approx 320 \text{ KB/token}$$

At sequence length 4096: $4096 \times 320 \text{ KB} \approx 1.25 \text{ GB}$ (per batch element!)

### PagedAttention

vLLM (Kwon et al., 2023) manages the KV cache in fixed-size **pages** (like virtual memory), enabling efficient batching and eliminating memory fragmentation.

In [ ]:
class KVCache:
    """Key-Value cache for transformer inference."""

    def __init__(self, num_layers, num_kv_heads, d_head, max_seq_len, batch_size=1, dtype=torch.float32):
        self.num_layers   = num_layers
        self.num_kv_heads = num_kv_heads
        self.d_head       = d_head
        self.max_seq_len  = max_seq_len
        self.batch_size   = batch_size
        self.dtype        = dtype

        # Pre-allocate cache tensors: list of (K, V) per layer
        # Shape: (batch_size, num_kv_heads, max_seq_len, d_head)
        self.cache = [
            (
                torch.zeros(batch_size, num_kv_heads, max_seq_len, d_head, dtype=dtype),
                torch.zeros(batch_size, num_kv_heads, max_seq_len, d_head, dtype=dtype)
            )
            for _ in range(num_layers)
        ]
        self.current_len = 0

    def update(self, layer_idx, new_k, new_v):
        """Append new K, V to cache at current position.
        Args:
            layer_idx: which layer
            new_k: (B, G, 1, d_head) new key
            new_v: (B, G, 1, d_head) new value
        Returns:
            full_k: (B, G, T_cur, d_head) all keys so far
            full_v: (B, G, T_cur, d_head) all values so far
        """
        t = self.current_len
        self.cache[layer_idx][0][:, :, t:t+1, :] = new_k
        self.cache[layer_idx][1][:, :, t:t+1, :] = new_v
        return self.get(layer_idx, t + 1)

    def step(self):
        """Advance time step."""
        self.current_len += 1

    def get(self, layer_idx, length=None):
        """Retrieve cached K, V up to given length."""
        if length is None:
            length = self.current_len
        k = self.cache[layer_idx][0][:, :, :length, :]
        v = self.cache[layer_idx][1][:, :, :length, :]
        return k, v

    def memory_bytes(self):
        bytes_per_element = {torch.float32: 4, torch.float16: 2, torch.bfloat16: 2}[self.dtype]
        total = (2 * self.num_layers * self.batch_size *
                 self.num_kv_heads * self.max_seq_len * self.d_head * bytes_per_element)
        return total


# Demo
cache = KVCache(num_layers=4, num_kv_heads=2, d_head=16, max_seq_len=32, batch_size=1)
print(f"KV Cache initialized: {cache.num_layers} layers, {cache.num_kv_heads} KV heads, d_head={cache.d_head}")
print(f"Pre-allocated memory: {cache.memory_bytes():,} bytes = {cache.memory_bytes()/1024:.1f} KB")

# Simulate 5 auto-regressive steps
for step in range(5):
    new_k = torch.randn(1, 2, 1, 16)   # (B, G, 1, d_head)
    new_v = torch.randn(1, 2, 1, 16)
    full_k, full_v = cache.update(layer_idx=0, new_k=new_k, new_v=new_v)
    cache.step()
    print(f"  Step {step+1}: cached K shape = {full_k.shape}, V shape = {full_v.shape}")

# Memory for various LLM configurations
print("\nKV Cache memory for production LLMs (BF16, batch_size=1):")
print(f"{'Model':>20} | {'L':>4} | {'G':>4} | {'d_k':>5} | {'T=512':>10} | {'T=4096':>10} | {'T=32K':>10}")
print("-" * 80)
configs = [
    ("LLaMA-3 8B",   32,  8, 128),
    ("LLaMA-3 70B",  80,  8, 128),
    ("Mixtral 8x7B", 32,  8, 128),
    ("GPT-4o (est)", 96, 16, 128),
]
for name, L, G, d_k in configs:
    def kv_gb(T): return 2 * L * G * d_k * T * 2 / 1e9  # BF16
    print(f"  {name:>20} | {L:>4} | {G:>4} | {d_k:>5} | {kv_gb(512):>9.3f}G | {kv_gb(4096):>9.3f}G | {kv_gb(32768):>9.3f}G")

## 6. Knowledge Distillation

Knowledge distillation (Hinton et al., 2015) trains a small **student** model to mimic a large **teacher** model.

### Combined Distillation Loss

$$\mathcal{L}_{KD} = (1 - \alpha)\mathcal{L}_{CE}(y, \hat{y}) + \alpha T^2 \mathcal{L}_{CE}\!\left(\sigma\!\left(\frac{z_T}{T}\right), \sigma\!\left(\frac{z_S}{T}\right)\right)$$

where:
- $z_T$, $z_S$ — teacher and student logits
- $T$ — temperature (softens the distributions; larger $T$ → softer targets)
- $\alpha \in [0, 1]$ — weight balancing hard labels vs soft labels
- $T^2$ — gradient scale compensation (since $\partial \mathcal{L}_{CE} / \partial z \propto 1/T$ but knowledge content scales as $1/T^2$)

### Why Soft Labels Help

Hard labels: $[0, 0, 1, 0, \ldots]$ — zero information about non-target classes.

Soft teacher labels: $[0.001, 0.003, 0.97, 0.002, \ldots]$ — encodes that class 2 and 4 are somewhat similar to class 3.

This "dark knowledge" in the teacher's probability distribution transfers generalizable structure to the student.

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    """Knowledge distillation loss combining hard and soft targets.
    Args:
        student_logits: (B, V) student model logits
        teacher_logits: (B, V) teacher model logits (no gradient)
        labels:         (B,)  hard ground-truth labels
        T:              temperature for softening
        alpha:          weight for soft targets
    Returns:
        loss:     combined loss scalar
        ce_loss:  cross-entropy component
        kd_loss:  soft-target distillation component
    """
    # Hard label CE loss
    ce_loss = F.cross_entropy(student_logits, labels)

    # Soft targets: temperature-scaled KL divergence
    teacher_soft = F.softmax(teacher_logits / T, dim=-1)
    student_log_soft = F.log_softmax(student_logits / T, dim=-1)

    # KL(teacher || student) = sum teacher * (log_teacher - log_student)
    # = CE(teacher_soft, student_soft) - H(teacher_soft)
    # Using F.kl_div: input=student_log_soft, target=teacher_soft
    kd_loss = F.kl_div(student_log_soft, teacher_soft, reduction='batchmean') * (T ** 2)

    loss = (1 - alpha) * ce_loss + alpha * kd_loss
    return loss, ce_loss, kd_loss


torch.manual_seed(42)
B, V = 8, 100

# Simulate teacher (large confident model) and student (smaller model)
teacher_logits = torch.randn(B, V) * 3.0   # more peaked distributions
student_logits = torch.randn(B, V)           # less peaked
labels = torch.randint(0, V, (B,))

print("Distillation Loss for varying temperature T:")
print(f"{'T':>4} | {'Total loss':>12} | {'CE loss':>10} | {'KD loss':>10} | {'Teacher entropy':>18}")
print("-" * 65)

for T in [1.0, 2.0, 4.0, 8.0, 16.0]:
    loss, ce, kd = distillation_loss(student_logits, teacher_logits, labels, T=T, alpha=0.5)
    # Teacher distribution entropy at this temperature
    teacher_soft = F.softmax(teacher_logits / T, dim=-1)
    entropy = -(teacher_soft * (teacher_soft + 1e-10).log()).sum(dim=-1).mean()
    print(f"{T:>4.1f} | {loss.item():>12.4f} | {ce.item():>10.4f} | {kd.item():>10.4f} | {entropy.item():>18.4f} nats")

print("\nHigher T -> softer teacher distribution -> higher entropy -> richer dark knowledge")

# Alpha effect
print("\nEffect of alpha (hard vs soft weight), T=4:")
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    loss, ce, kd = distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=alpha)
    print(f"  α={alpha:.2f}: total={loss.item():.4f}  CE={ce.item():.4f}  KD={kd.item():.4f}")

## 7. Magnitude Pruning

Pruning removes parameters that contribute little to model output, reducing model size and (with sparse compute) inference cost.

### Unstructured Magnitude Pruning

Zero out all weights with absolute value below threshold $\tau$:

$$\hat{w}_{ij} = \begin{cases} w_{ij} & \text{if } |w_{ij}| \geq \tau \\ 0 & \text{otherwise} \end{cases}$$

Sparsity is: $s = \frac{\|\hat{\mathbf{W}}\|_0}{N} \times 100\%$ (percent of zero weights).

### Structured Pruning

Remove entire neurons, heads, or layers:
- **Head pruning:** Remove attention heads with low importance scores
- **Neuron pruning:** Remove FFN neurons with low $L_1$ norm
- **Layer dropping:** Remove entire transformer layers (used in LLaMA pruning)

Structured pruning enables **dense computation** on the pruned model (no sparse kernels needed).

### Lottery Ticket Hypothesis

Frankle & Carlin (2019): Within a large random network there exist small **subnetworks** ("winning tickets") that, when trained in isolation from the same initialization, reach full accuracy. This justifies iterative magnitude pruning.

In [ ]:
import torch.nn.utils.prune as prune

torch.manual_seed(42)

# Create a simple linear layer to prune
layer = nn.Linear(128, 64)
print(f"Original layer: {layer.weight.shape}")
print(f"Total weights: {layer.weight.numel():,}")
print(f"Initial sparsity: {(layer.weight == 0).float().mean().item():.1%}")

# Apply L1 unstructured pruning at various sparsity levels
print("\nMagnitude Pruning (L1 unstructured):")
print(f"{'Sparsity':>10} | {'Nonzero':>10} | {'Weight L2 norm':>16} | {'Output error vs dense':>22}")
print("-" * 70)

x_test = torch.randn(4, 128)
out_dense = layer(x_test).detach()

for amount in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    # Fresh copy of layer
    layer_copy = nn.Linear(128, 64)
    layer_copy.weight.data = layer.weight.data.clone()
    layer_copy.bias.data   = layer.bias.data.clone()

    if amount > 0:
        prune.l1_unstructured(layer_copy, name='weight', amount=amount)
        prune.remove(layer_copy, 'weight')  # make mask permanent

    sparsity    = (layer_copy.weight == 0).float().mean().item()
    nonzero     = (layer_copy.weight != 0).sum().item()
    weight_norm = layer_copy.weight.norm(2).item()
    out_pruned  = layer_copy(x_test).detach()
    out_error   = (out_pruned - out_dense).norm(2).item() / out_dense.norm(2).item()

    print(f"{sparsity:>10.1%} | {nonzero:>10,} | {weight_norm:>16.4f} | {out_error:>22.4f}")

# Structured pruning: remove 50% of output neurons with smallest L1 norms
print("\nStructured Pruning (remove neurons with lowest L1 norm):")
layer_structured = nn.Linear(128, 64)
layer_structured.weight.data = layer.weight.data.clone()

neuron_norms = layer_structured.weight.abs().sum(dim=1)   # L1 norm per output neuron
k_keep = 32  # keep top-32 neurons
_, top_k_idx = neuron_norms.topk(k_keep)

# Create pruned weight
pruned_weight = layer_structured.weight[top_k_idx, :]   # (k_keep, d_in)
print(f"  Original: {layer_structured.weight.shape} -> Pruned: {pruned_weight.shape}")
print(f"  Parameters: {layer_structured.weight.numel():,} -> {pruned_weight.numel():,}  ({k_keep/64:.0%} remaining)")
print(f"  This gives a true 2x speedup with no sparse kernel needed.")

## 8. Efficient Inference Stack

Production LLM inference combines all the techniques from this chapter:

| Technique | Memory Saving | Speed Gain |
|---|---|---|
| FlashAttention | $O(T)$ vs $O(T^2)$ HBM reads | 2-4x attention speed |
| GQA | $H/G$ KV cache reduction | Higher batch sizes |
| INT8 quantization | 2x weight memory | 1.5-2x throughput |
| Speculative decoding | None | 2-4x generation speed |
| Continuous batching | Eliminates padding waste | 2-10x throughput |
| KV cache management | Efficient allocation | Enables longer contexts |

### Throughput Formula

$$\text{Throughput (tokens/sec)} = \frac{\text{batch\_size} \times \text{tokens\_generated}}{\text{wall\_clock\_time}}$$

For memory-bandwidth-bound inference (typical at batch_size=1):

$$\text{Throughput} \approx \frac{\text{HBM bandwidth}}{\text{bytes per token}} = \frac{\text{HBM bandwidth}}{2 \times N_{\text{params}} \times \text{dtype\_bytes}}$$

A100 80GB: 2 TB/s bandwidth. LLaMA-3 8B BF16 (16 GB): ~62.5 tokens/sec theoretical.

In [ ]:
torch.manual_seed(42)

def efficient_attention_block(x, num_heads=8, num_kv_heads=2, use_flash=True):
    """Efficient attention using F.scaled_dot_product_attention (FlashAttention kernel)
    combined with Grouped Query Attention.
    Args:
        x: (B, T, d_model)
    Returns:
        output: (B, T, d_model)
    """
    B, T, d_model = x.shape
    d_head = d_model // num_heads
    num_groups = num_heads // num_kv_heads

    # QKV projections (weight matrices created inline for demo)
    W_q = torch.randn(d_model, num_heads * d_head, device=x.device, dtype=x.dtype) * 0.02
    W_k = torch.randn(d_model, num_kv_heads * d_head, device=x.device, dtype=x.dtype) * 0.02
    W_v = torch.randn(d_model, num_kv_heads * d_head, device=x.device, dtype=x.dtype) * 0.02
    W_o = torch.randn(d_model, d_model, device=x.device, dtype=x.dtype) * 0.02

    Q = (x @ W_q).view(B, T, num_heads,    d_head).transpose(1, 2)    # (B, H, T, d_head)
    K = (x @ W_k).view(B, T, num_kv_heads, d_head).transpose(1, 2)    # (B, G, T, d_head)
    V = (x @ W_v).view(B, T, num_kv_heads, d_head).transpose(1, 2)    # (B, G, T, d_head)

    # GQA: expand K,V
    K = K.repeat_interleave(num_groups, dim=1)   # (B, H, T, d_head)
    V = V.repeat_interleave(num_groups, dim=1)   # (B, H, T, d_head)

    # FlashAttention kernel (uses O(T) HBM reads via tiling, is_causal=True)
    attn_out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

    # Merge heads
    attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, d_model)
    return attn_out @ W_o


# Benchmark: measure tokens/sec for different batch sizes
d_model = 128
T = 64    # sequence length
num_tokens_generated = 50
dtype = torch.bfloat16

print("Inference Throughput Benchmark (FlashAttention + GQA, BF16):")
print(f"d_model={d_model}, seq_len={T}, tokens_to_generate={num_tokens_generated}")
print(f"{'Batch size':>12} | {'Wall time (s)':>14} | {'Total tokens':>14} | {'Tokens/sec':>12}")
print("-" * 60)

for batch_size in [1, 4, 16]:
    x = torch.randn(batch_size, T, d_model, dtype=dtype)

    # Warmup
    for _ in range(3):
        _ = efficient_attention_block(x)

    # Timed run: simulate generating num_tokens_generated tokens
    start = time.perf_counter()
    for step in range(num_tokens_generated):
        # Simulate one decode step: process single new token with cached context
        x_new = torch.randn(batch_size, 1, d_model, dtype=dtype)
        # In real inference, x_ctx would be combined with kv cache;
        # here we simulate the attention computation cost
        context = torch.cat([x[:, :T//2, :], x_new], dim=1)  # T//2 + 1 tokens
        out = efficient_attention_block(context)

    elapsed = time.perf_counter() - start
    total_tokens = batch_size * num_tokens_generated
    tps = total_tokens / elapsed

    print(f"{batch_size:>12} | {elapsed:>14.4f} | {total_tokens:>14,} | {tps:>12.1f}")

print()
print("Note: CPU benchmark — GPU (A100) would be 100-1000x faster.")
print("The scaling with batch_size shows the benefit of batching (amortized memory bandwidth).")

## 9. Summary: Math → Engineering

Every efficiency technique in this chapter is a mathematical insight, carefully implemented:

| Technique | Mathematical Insight | Memory Gain | Speed Gain |
|---|---|---|---|
| **MoE** | Conditional computation: $\sum_{i \in \text{TopK}} g_i E_i(x)$ | $N\times$ capacity at $k/N$ compute | Constant FLOPs with larger model |
| **GQA** | $G$ shared KV heads: $G/H$ KV memory | $H/G\times$ KV reduction | Higher batch size at same memory |
| **INT8 Quant** | $w \approx s \cdot \text{round}(w/s)$, $|\text{error}| \leq s/2$ | $2-4\times$ weight memory | $1.5-2\times$ via BW reduction |
| **Speculative Decoding** | Rejection sampling: accept iff $u \leq p/q$ | None | $\gamma\times$ when $p \approx q$ |
| **KV Cache** | Memoize $K, V$ for past tokens | $O(T)$ per decode step | $O(1)$ incremental attention |
| **Distillation** | Soft labels encode similarity structure | Smaller student model | Training efficiency + smaller model |
| **Pruning** | Zero small weights: $|w_{ij}| < \tau$ | Proportional to sparsity | Dense speedup (structured) |
| **FlashAttention** | Tiled HBM IO: $O(T)$ reads vs $O(T^2)$ | Same FLOPs, less HBM | $2-4\times$ wall-clock attention |

### Closing Remark

> **Every efficiency technique is a mathematical insight — the engineering is just careful implementation.**

The journey from a research idea to a production LLM serving millions of users involves:

1. **Mathematical formulation** — articulate the insight precisely (e.g., sparse conditional computation for MoE)
2. **Numerical analysis** — quantify the error/quality tradeoff (e.g., INT8 quantization error bounds)
3. **Systems implementation** — fused CUDA kernels, memory management, batching strategy
4. **Empirical validation** — verify that quality holds at scale (e.g., GQA gives same quality as MHA)

The models powering today's AI applications are the product of thousands of such insights — from the original Attention is All You Need paper through FlashAttention, LoRA, RLHF, DPO, MoE, and speculative decoding. Understanding the mathematics behind each technique is what enables you to innovate at this frontier.

**This concludes the Mathematics Behind LLMs book series.**